# CANOPUS - Download and Extract Riometer Data

***

**Tutorial:** This tutorial explains how to extract CANOPUS riometer data from the open data portal.  
**Mission and Instrument:** CANOPUS (Canadian Auroral Network for the OPEN Program Unified Study)  
**Astronomical Target:** Measure Earth's magnetic field to study space weather events like geomagnetic storms and substorms.  
**System Requirements:** Access to the internet.  
**Tutorial Level:** Intermediate

The CANOPUS data can be found in both CSV and raw datasets on the CSA Open Data Portal. The raw data can be found [here](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/CANOPUS/) and the CSV data can be found [here](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/CANOPUS_CSV/).

# Part 1: Downloading CANOPUS data

The CARISMA/CANOPUS data is hosted on the [CSA Open Data Portal](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub). Files can be downloaded by web scraping. 

# 1.1 Web Scraping

### Directory Structure 

```
/users/OpenData_DonneesOPuvertes/pub/
|
|-- carisma_csv/                                    <-- CARISMA magnetometer data (csv)
|
|-- CANOPUS_CSV/                                    <-- CANOPUS riometer data (csv)
|   +-- old_canopus_riometer_format/                <-- Old riometer format
|
|-- carisma/                                        <-- RAW CARISMA files (.tar)
|
|-- CANOPUS/                                        <-- Raw CANOPUS files (.tar.gz)
```




### Exploring the Server

In [ ]:
# %pip install pandas matplotlib

In [ ]:
import os 
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from io import StringIO

# The CSA server will throw a self-signed certificate error, so we need to disable SSL verification
requests.packages.urllib3.disable_warnings()

BASE_URL = "https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/"
RIO_BASE = BASE_URL + "CANOPUS/CANOPUS_CSV/"

In [ ]:
def list_directory(url):
    resp = requests.get(url, verify=False)
    
    links = re.findall(r'href="([^"]+)"', resp.text)

    # Filter parent-directory links, query strings
    names = []
    for link in links:
        name = link.strip('/').split('/')[-1]
        if name and name != ".." and "?" not in link and link != "../":
            # Skip links that point to parent path
            if not link.startswith("/"):
                names.append(name)
    return names    


# Browse the riometer data archive
print("Available years of riometer data:")
years = list_directory(RIO_BASE)
print(years)

# List stations for 2008
print("\nAvailable stations for 2007:")
stations = list_directory(RIO_BASE + "2007/")
print(stations)

# List first few files for station GILL
print("\nFirst 5 files for station GILL in 2007:")
files = list_directory(RIO_BASE + "2007/GILL/")
print(files[:5])


### Part 1: Download Riometer Data

In addition to magnetometer data (see the CARISMA Magnetometer Data tutorial), CANOPUS also collected RIOMETER data.

#### 1.1 Download Programatically

In [ ]:
def download_rio_file(station, date_str, save_dir="data/rio"):
    year = date_str[:4]
    remote_dir = f"{RIO_BASE}{year}/{station}/"

    os.makedirs(save_dir, exist_ok=True)

    try:
        # Scrape the directory tlisting to find matching files
        files = list_directory(remote_dir)
        matching_files = [f for f in files if date_str in f]

        if not matching_files:
            print(f"No rioemeter files found for {station} on {date_str}")
            return None
        
        filename = matching_files[0]
        file_url = remote_dir + filename
        local_path = os.path.join(save_dir, filename)

        resp = requests.get(file_url, verify=False)
        with open(local_path, 'wb') as f:
            f.write(resp.content)
        print(f"Downloaded {filename} to directory: {local_path}")
        return local_path
    
    except Exception as e:
        print(f"Error downloading file for {station} on {date_str}: {e}")
        return None

# Example usage: Download RIO file for GILL station on January 6, 1999
rio_file = download_rio_file("GILL", "19990106")

#### 1.2 Manual Download

1. First visit the [CSA Open Data Portal - CARISMA Dataset (CSV)](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/mag/daily/)
2. Browse or search the files you need
3. Download and save them locally
4. Place the files into the /data/rio/ folder of this tutorial

### Part 2: Load and Explore Riometer Data

The magnetometer CSV files have a specific structure with comment lines, station metadata, and actual data. 

**What the files look like:**
```
#CANOPUS Riometer Data ----  gil     19990106  lat:56.4 long:265.4
#Baseline Version 1  ---  Summary Data has not been checked for errors
"#contact:  Eric Donovan (edonovan@ucalgary.ca), Emma Spanswick (elspansw@ucalgary.ca)"
#----------------------------------------------
 
date(dd/mm/yy),time (UT),Absorption (dB),Raw Signal (Volts)
06/01/99,00:01:53,0.009,1.573
06/01/99,00:01:58,0.292,1.487
06/01/99,00:02:02,0.016,1.570
```

The key fields are:
- **Absorption (dB)**: Cosmic noise absorption -- higher values mean more ionospheric activity
- **Raw Signal (Volts)**: The raw voltage measurement from the riometer


In [ ]:
def load_rio_data(file_path):
    data_lines = []
    header_found = False

    with open(file_path, 'r') as f:
        for line in f:
            stripped = line.strip().strip('"')

            # Skip comment lines and blank lines
            if stripped.startswith("#") or stripped == "":
                continue
            
            # Detect station metadata line (no commans, space separated)
            if not header_found and 'date' in stripped.lower():
                header_found = True
                continue

            # Data line
            data_lines.append(stripped)

    # Convert data lines to DataFrame
    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        names=["date", "time", "absorption_dB", "raw_signal_V"],
        header=None
    )

    # Combine date and time into a single datetime column
    df['datetime'] = pd.to_datetime(
        df['date'] + ' ' + df['time'], 
        format='%d/%m/%y %H:%M:%S'
    )

    return df

In [ ]:
# Load and explore one date of riometer data 
rio_df = load_rio_data(rio_file)

print(f"\nRiometer dataframe shape: {rio_df.shape}")
print(f"Time range: {rio_df['datetime'].min()} to {rio_df['datetime'].max()}")
print(f"\nAbsorption (dB) stats:")
print(rio_df[['absorption_dB', 'raw_signal_V']].describe().round(3))

rio_df.head()

### Part 3: Visualize Riometer Data

Plot riometer data: absorption and raw signal

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(rio_df['datetime'], rio_df['absorption_dB'], color='purple', linewidth=0.5)
axes[0].set_ylabel('Absorption (dB)')
axes[0].set_title(f'CANOPUS Riometer Data - GILL Station on {rio_df["datetime"].dt.date.iloc[0]}')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rio_df['datetime'], rio_df['raw_signal_V'], color='teal', linewidth=0.5)
axes[1].set_ylabel('Raw Signal (V)')
axes[1].set_xlabel('Time (UTC)')
axes[1].grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=3))

plt.tight_layout()
plt.savefig("riometer_plot.png", dpi=150, bbox_inches='tight')
plt.show()

print("Riometer plot saved as 'riometer_plot.png'.")
